# Bayesian Core Analysis: Genişletilmiş Model Denemeleri

Bu notebook, hipotez testlerinde her zaman anlamlı çıkmayan ancak daha güçlü korelasyon veya pratik etki büyüklüğü nedeniyle karşılaştırmayı hak eden değişkenler için daha geniş bir deneme alanı tutar.

Buradaki modeller, 03 numaralı notebook'taki tercih edilen modelin yanında
ikincil karşılaştırma noktaları olarak kullanılır.


In [ ]:
# --- Kütüphaneler ---
from pathlib import Path

import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from IPython.display import display, Markdown


In [ ]:
# --- Veri yükleme ve Bayesian değişken inşası ---
# Bu blok, 01_bayesian_descriptive_and_normality.ipynb ile birebir aynı
# MAP selection mantığını kullanır. Böylece tüm notebook'larda
# obs_value_bayes_selected_signed_log aynı yöntemle üretilmiş olur.

project_root = Path.cwd()
while not (project_root / 'data').exists() and project_root != project_root.parent:
    project_root = project_root.parent

data_path = project_root / 'data' / 'processed' / 'OECD_FDI_Except_Pointless.csv'
df = pd.read_csv(data_path, sep=';', encoding='latin1', engine='python', on_bad_lines='skip')

# Ham sayısal değer — nokta binlik ayracı olarak kullanılmış, kaldırılıyor
df['OBS_VALUE_NUM'] = pd.to_numeric(
    df['OBS_VALUE'].astype(str).str.replace('.', '', regex=False),
    errors='coerce'
)

# Eksik değerleri global medyanla doldur; signed-log dönüşümü uygula
df['obs_value_imputed']    = df['OBS_VALUE_NUM'].fillna(df['OBS_VALUE_NUM'].median())
df['obs_value_signed_log'] = np.sign(df['obs_value_imputed']) * np.log1p(np.abs(df['obs_value_imputed']))

# --- Empirical Bayes shrinkage: grup düzeyinde posterior ortalamalar ---
global_mean = df['obs_value_signed_log'].mean()
global_var  = df['obs_value_signed_log'].var(ddof=1)

def empirical_bayes_mean(data, group_col, value_col='obs_value_signed_log'):
    grouped  = data.groupby(group_col)[value_col]
    stats_df = grouped.agg(['mean', 'count', 'var']).reset_index()
    between_var = max(stats_df['mean'].var(ddof=1), 1e-6)
    within_var  = max(stats_df['var'].fillna(global_var).mean(), 1e-6)
    stats_df['shrinkage_weight'] = (
        (stats_df['count'] * between_var) /
        (stats_df['count'] * between_var + within_var)
    )
    stats_df['posterior_mean'] = (
        stats_df['shrinkage_weight'] * stats_df['mean'] +
        (1 - stats_df['shrinkage_weight']) * global_mean
    )
    return stats_df[[group_col, 'posterior_mean']]

for group_col, new_col in [
    ('REF_AREA',    'bayes_ref_area'),
    ('ACTIVITY',    'bayes_activity'),
    ('TYPE_ENTITY', 'bayes_type_entity'),
]:
    post = empirical_bayes_mean(df, group_col).rename(columns={'posterior_mean': new_col})
    df   = df.merge(post, on=group_col, how='left')

df['bayes_global'] = global_mean

# --- MAP (Maximum A Posteriori) seçimi ---
sigma2         = max(df['obs_value_signed_log'].var(ddof=1), 1e-6)
candidate_cols = ['bayes_ref_area', 'bayes_activity', 'bayes_type_entity', 'bayes_global']

for col in candidate_cols:
    df[f'loglik_{col}'] = -0.5 * ((df['obs_value_signed_log'] - df[col]) ** 2) / sigma2

loglik_cols = [f'loglik_{col}' for col in candidate_cols]
max_loglik  = df[loglik_cols].max(axis=1)
exp_shifted = np.exp(df[loglik_cols].sub(max_loglik, axis=0))
posterior_probs = exp_shifted.div(exp_shifted.sum(axis=1), axis=0)
posterior_probs.columns = [c.replace('loglik_', 'posterior_prob_') for c in posterior_probs.columns]
df = pd.concat([df, posterior_probs], axis=1)

df['bayes_best_source'] = (
    posterior_probs.idxmax(axis=1)
    .str.replace('posterior_prob_', '', regex=False)
)

df['obs_value_bayes_selected_signed_log'] = np.select(
    [
        df['bayes_best_source'] == 'bayes_ref_area',
        df['bayes_best_source'] == 'bayes_activity',
        df['bayes_best_source'] == 'bayes_type_entity',
    ],
    [df['bayes_ref_area'], df['bayes_activity'], df['bayes_type_entity']],
    default=df['bayes_global']
)

# --- Genişletilmiş deneme değişkenleri ---
# En sık gözlemlenen 10 aktivite kodu
df['activity_top10_flag'] = df['ACTIVITY'].isin(
    df['ACTIVITY'].value_counts().head(10).index
).astype(int)

# En sık gözlemlenen 10 ülke kodu
df['country_top10_flag'] = df['REF_AREA'].isin(
    df['REF_AREA'].value_counts().head(10).index
).astype(int)

# TYPE_ENTITY x MEASURE_PRINCIPLE etkileşim değişkeni
# Not: bu iki değişken zaten modelde ayrı ayrı yer aldığından
# string concatenation proxy yerine C(TYPE_ENTITY):C(MEASURE_PRINCIPLE)
# formulasyonu multicollinearity riskini daha iyi yönetir
df['type_meas_interaction_flag'] = (
    df['TYPE_ENTITY'].astype(str) + '_' + df['MEASURE_PRINCIPLE'].astype(str)
)

display(Markdown(f'**Veri boyutu:** {df.shape[0]:,} satır'))
df.head()


In [ ]:
# --- Deneme formülleri ---
# Trial A: temel model — yalnızca ana kategorik değişkenler ve zaman
# Trial B: aktivite flag'i eklendi — aktivite gruplarının ayrışma gücü test ediliyor
# Trial C: ülke flag'i eklendi — büyük ekonomilerin ayrı etkisi var mı?
# Trial D: etkileşim proxy'si eklendi — TYPE_ENTITY ve MEASURE_PRINCIPLE birlikte farklı mı davranıyor?
trial_formulas = {
    'Trial_A_core': (
        'obs_value_bayes_selected_signed_log ~ '
        'C(TYPE_ENTITY) + C(MEASURE_PRINCIPLE) + TIME_PERIOD'
    ),
    'Trial_B_plus_activity_flag': (
        'obs_value_bayes_selected_signed_log ~ '
        'C(TYPE_ENTITY) + C(MEASURE_PRINCIPLE) + TIME_PERIOD + activity_top10_flag'
    ),
    'Trial_C_plus_country_flag': (
        'obs_value_bayes_selected_signed_log ~ '
        'C(TYPE_ENTITY) + C(MEASURE_PRINCIPLE) + TIME_PERIOD + country_top10_flag'
    ),
    'Trial_D_plus_interaction_proxy': (
        'obs_value_bayes_selected_signed_log ~ '
        'C(TYPE_ENTITY) + C(MEASURE_PRINCIPLE) + TIME_PERIOD + C(type_meas_interaction_flag)'
    ),
}

trial_rows = []
for name, formula in trial_formulas.items():
    model = smf.ols(formula, data=df).fit()
    trial_rows.append({
        'model':    name,
        'aic':      model.aic,
        'bic':      model.bic,
        'rsquared': model.rsquared,
    })

trial_df = pd.DataFrame(trial_rows).sort_values(['aic', 'bic'])
trial_df


### Yorum

Bu deneme notebook'u, katı hipotez testi filtresinden her zaman geçemeyen
ancak pratik önemi, korelasyon gücü ve etki büyüklüğü nedeniyle
karşılaştırma setinde kalan aday değişkenleri bilinçli olarak tutar.

Trial D'deki `type_meas_interaction_flag`, TYPE_ENTITY ve MEASURE_PRINCIPLE'ın
string birleştirmesidir. Bu iki değişken modelde zaten ayrı ayrı yer aldığından
multicollinearity riski taşır; AIC/BIC karşılaştırmasında bu nokta göz önünde bulundurulmalıdır.
